In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
import os
os.environ['HF_TOKEN'] = 'your token'

In [24]:
# pip install accelerate

from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import requests
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Using AutoModelForCausalLM for more general model loading
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16
).eval()

processor = AutoProcessor.from_pretrained(model_id)

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

# NOTE: Llama-3.2-3B-Instruct does not support multimodal inputs.
# I will remove the image and only use the text input for the prompt.
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Describe this image in detail."
    }
]


inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generation = generation[0][input_len:]

decoded = processor.decode(generation, skip_special_tokens=True)
print(decoded)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I don't see an image. Can you please provide the image you'd like me to describe, or describe it to me in words so I can help you with your request?


# Task
Adapt the provided Python code to use the `google/gemma-3-4b-it` model for classification, process data from the input CSV file "/home/smaniyar_umass_edu/BioNLP_Ontology/nlp/privacyQA/data/PrivacyQA_EMNLP/sampled_data/stratified_test_sample_1000.csv", and save the results to the output CSV file "/home/smaniyar_umass_edu/BioNLP_Ontology/nlp/privacyQA/test_results/qwen_2_5_privacy_qa.csv".

## Set up environment

### Subtask:
Ensure necessary libraries (`transformers`, `pandas`, `torch`) are installed and the Hugging Face token is set up (either via Secrets Manager or `huggingface_hub.login`).


**Reasoning**:
The subtask requires ensuring necessary libraries are installed and the Hugging Face token is set up. The notebook already has cells for installing transformers and logging in with Hugging Face. I will add a cell to install pandas and torch if they are not already installed.



## Load data

### Subtask:
Load the input CSV file into a pandas DataFrame. The user will need to upload the file to Colab first.


**Reasoning**:
The goal is to load the input CSV file into a pandas DataFrame. The user has specified the file path. This requires importing pandas, defining the file path, reading the CSV, and displaying the head and info to confirm.



## Define processing logic

### Subtask:
Adapt the `make_prompt` and `extract_reasoning_and_label` functions from the user's provided code snippet.


**Reasoning**:
Define the `make_prompt` and `extract_reasoning_and_label` functions. The `make_prompt` function will construct a prompt for the Gemma model using the text and question from a DataFrame row. The `extract_reasoning_and_label` function will parse the model's output to find the reasoning and the predicted label.



**Reasoning**:
Iterate through the dataframe rows, generate prompts, run inference, and extract reasoning and labels.



## Iterate and infer

### Subtask:
Retry iterating through each row of the DataFrame, create the appropriate prompt, run inference using the loaded Gemma model, and extract the reasoning and label. This retry addresses the previous failure where the DataFrame was not loaded.


**Note:** The following code snippet uses the `vllm` library and file paths that are likely not accessible in this Colab environment. Running this code as is may result in errors.

In [27]:
import pandas as pd
#from vllm import LLM, SamplingParams
import json
import re
import time


# =========================================================
# === 0. USER-DEFINED PATHS ===============================
# =========================================================
INPUT_FILE = "/content/test_data/stratified_test_sample_1000.csv"
OUTPUT_FILE = "/content/test_results/llama_3b_privacy_qa.csv"
LOG_FILE = "/content/test_results/llama_3b_privacy_qa.log"
# =========================================================


# === 1. Load dataset ===
# This will likely fail due to FileNotFoundError
df = pd.read_csv(INPUT_FILE)

# === 2. Create unique ID ===
df["id"] = (
    df["DocID"].astype(str) + "_" +
    df["QueryID"].astype(str) + "_" +
    df["SentID"].astype(str)
)

def make_prompt(query: str, segment: str) -> str:
    return f"""
INSTRUCTION: Classify if the segment contains information relevant to the query.
RELEVANCE CRITERIA:
- "Relevant": Segment directly addresses or contains information about the query topic
- "Irrelevant": Segment does not mention or relate to the query topic
OUTPUT: JSON only with 'reasoning' and 'final_label' fields.

QUERY: {query}
SEGMENT: {segment}

RESPONSE:
""".strip()




def extract_reasoning_and_label(output_text: str):
    output_text = output_text.strip()

    # Try to isolate the JSON part
    match = re.search(r"\{.*\}", output_text, re.DOTALL)
    if not match:
        return "No JSON found in output", "Error"

    json_str = match.group(0)

    # Try parsing JSON safely
    try:
        data = json.loads(json_str)
        reasoning = data.get("reasoning")
        final_label = data.get("final_label")
        return reasoning, final_label
    except json.JSONDecodeError:
        # Fallback: regex extraction if JSON is malformed
        try:
            reasoning_match = re.search(r'"reasoning":\s*"([^"]*)"', output_text)
            label_match = re.search(r'"final_label":\s*"([^"]*)"', output_text)

            reasoning = reasoning_match.group(1) if reasoning_match else "Unable to extract reasoning"
            final_label = label_match.group(1) if label_match else "Error"

            return reasoning, final_label
        except Exception:
            return "JSON parsing failed", "Error"

In [29]:
# === 6. Prepare dataframe columns ===
df["model_output"] = None
df["reasoning"] = None
df["final_label"] = None

# === 7. Initialize log file ===
with open(LOG_FILE, "w", encoding="utf-8") as log:
    log.write("========================================\n")
    log.write("LLAMA 3 Privacy QA Classification Log\n")
    log.write("========================================\n\n")

# === 8. Sequential processing ===
if 'model' in locals() and 'processor' in locals() and model is not None and processor is not None:
    for i, row in df.iterrows():
        query = str(row["Query"])
        segment = str(row["Segment"])
        doc_id = row["id"]
        prompt = make_prompt(query, segment)

        print(f"\n--- Processing {i+1}/{len(df)} ---")
        print(f"ID: {doc_id}")
        print(f"Query: {query}")

        try:
            # Tokenize and send to model
            inputs = processor(text=prompt, return_tensors="pt").to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=200,
                    temperature=0.5,
                    top_p=0.9,
                    do_sample=True,
                )

            raw_output = processor.decode(outputs[0], skip_special_tokens=True).strip()
            # print("raw output:", raw_output)

            # UNCOMMENT THIS CRITICAL PART:
            reasoning, final_label = extract_reasoning_and_label(raw_output)
            print(f"Reasoning: {reasoning}")
            print(f"Final Label: {final_label}")

            # Store results
            df.at[i, "model_output"] = raw_output
            df.at[i, "reasoning"] = reasoning
            df.at[i, "final_label"] = final_label

            # Log results
            with open(LOG_FILE, "a", encoding="utf-8") as log:
                log.write(f"ID: {doc_id}\n")
                log.write(f"PROMPT:\n{prompt}\n")
                log.write(f"RAW OUTPUT:\n{raw_output}\n")
                log.write(f"REASONING: {reasoning}\n")
                log.write(f"FINAL LABEL: {final_label}\n")
                log.write("------------------------------------------------------------\n\n")

            print(f"Processed {i+1}/{len(df)} | Label: {final_label}")

            # Optional pause to avoid GPU overload
            # time.sleep(0.5)

        except Exception as e:  # FIXED INDENTATION
            print(f"Error processing row {i}: {e}")
            df.at[i, "model_output"] = f"Error: {e}"
            df.at[i, "reasoning"] = f"Error during processing: {e}"
            df.at[i, "final_label"] = "Error"

            with open(LOG_FILE, "a", encoding="utf-8") as log:
                log.write(f"ID: {doc_id}\nError: {e}\n")
                log.write("------------------------------------------------------------\n\n")

    # === 9. Save results ===
    df.to_csv(OUTPUT_FILE, index=False)
    print("\n✅ Done!")
    print(f"Results saved to: {OUTPUT_FILE}")
    print(f"Log saved to: {LOG_FILE}")
    display(df[["id", "reasoning", "final_label"]].head())

else:
    print("❌ Model or processor not loaded. Inference skipped.")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



--- Processing 1/999 ---
ID: 23andMe _7_23andMe _7_46_23andMe _7_46_328
Query: do you have any association with google?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Reasoning: This segment does not mention or relate to the query topic
Final Label: Irrelevant
Processed 1/999 | Label: Irrelevant

--- Processing 2/999 ---
ID: 23andMe _7_23andMe _7_29_23andMe _7_29_332
Query: does 23andme use my test results for marketing purposes?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Reasoning: Irrelevant
Final Label: Irrelevant
Processed 997/999 | Label: Irrelevant

--- Processing 998/999 ---
ID: Keep _2_Keep _2_23_Keep _2_23_45
Query: is any information shared with third parties?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Reasoning: The segment mentions sharing of personal information, which is related to the query about sharing information with third parties.
Final Label: Relevant
Processed 998/999 | Label: Relevant

--- Processing 999/999 ---
ID: 23andMe _7_23andMe _7_26_23andMe _7_26_101
Query: will my test results be shared with any third party entities?
Reasoning: The segment mentions that 23andMe may share summary statistics, which do not identify any particular individual or contain individual-level information, with qualified research collaborators. This is relevant to the query about whether test results will be shared with any third party entities.
Final Label: relevant
Processed 999/999 | Label: relevant

✅ Done!
Results saved to: /content/test_results/llama_3b_privacy_qa.csv
Log saved to: /content/test_results/llama_3b_privacy_qa.log


,id,reasoning,final_label
0,23andMe _7_23andMe _7_46_23andMe _7_46_328,This segment does not mention or relate to the...,Irrelevant
1,23andMe _7_23andMe _7_29_23andMe _7_29_332,The segment does not mention or relate to the ...,Irrelevant
2,Viber Messenger _8_Viber Messenger _8_41_Viber...,The segment does not mention or relate to the ...,Irrelevant
3,Fiverr _1_Fiverr _1_16_Fiverr _1_16_120,Irrelevant,Irrelevant
4,Keep _2_Keep _2_18_Keep _2_18_80,The segment does not mention or relate to the ...,Irrelevant
